# 09 · Mechanistic interpretability of a toy model

**Paper:** "The Asymmetric Superposition Motif" (pp. 38–39), with the absolute-value model of
"Computation in Superposition" (pp. 34–38) and footnote 15.
**Experiment:** E09 `mech_interp_abs`. **Library:** `superposition.interp`.

**Learning goals**

1. Read a neuron through its *path gains* |W₁[h,i]|·W₂[j,h]: what it computes, what it
   interferes with, what it inhibits.
2. Name every neuron of a small |x| model: positive and negative sides, (asymmetric)
   superposition, inhibitors.
3. Test a story with **ablations**: if a role is right, zeroing the neuron hurts exactly the
   features the role says it serves.
4. Recognize the **asymmetric superposition + inhibition** motif of p. 39, and measure how much
   of the model a hand-built **rebuild** recovers.

The quick profile of E09 takes about 5 minutes on a laptop CPU.

In [ ]:
# Runs locally and in Colab. In a fresh environment (e.g. Colab), install the package first.
# Replace <OWNER> with the GitHub owner of your copy of the repository.
try:
    import superposition  # noqa: F401
except ImportError:
    %pip install -q "git+https://github.com/<OWNER>/superposition.git"

In [ ]:
import pandas as pd
import torch
import yaml
from IPython.display import Image, display

from superposition import interp
from superposition.experiments.runner import run_experiment
from superposition.runs import list_runs
from superposition.utils import make_generator

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

REUSE = True  # reuse your latest quick-profile run if there is one; False always re-runs


def quick_run(name: str):
    # Your latest complete quick-profile run of `name` with its figures, or a fresh one. Runs
    # made without figures (`superposition run --no-figures`, or the replication tests) are
    # skipped: this notebook shows the run's figures.
    if REUSE:
        for run in reversed(list_runs(name)):
            quick = yaml.safe_load(run.config_path.read_text())["profile"] == "quick"
            if quick and any(run.figures_dir.glob("*.png")):
                print(f"reusing run {run.run_id}")
                return run
    return run_experiment(name, "quick", device="auto")


def criteria(run) -> pd.DataFrame:
    doc = run.read_acceptance()
    print("overall:", doc["overall"], f"({doc['n_passed']}/{doc['n_graded']} graded criteria)")
    return pd.DataFrame(doc["criteria"])[["id", "status", "value"]]

## 1. Path gains: reading one neuron

The model is h = ReLU(W₁x), y′ = ReLU(W₂h + b), with target |x| (p. 35). When only feature i is
active, xᵢ = ±v, neuron h fires only if the sign of xᵢ matches W₁[h, i], and then adds
v·|W₁[h,i]|·W₂[j,h] to output j. That number per unit v is the **path gain** G_h[i, j].

Start with the model p. 36 describes: for each feature a positive-side neuron ReLU(xᵢ) and a
negative-side neuron ReLU(−xᵢ), both writing 1 to output i.

**Predict:** what are the roles, and how much does feature 0's loss rise if you zero its
positive-side neuron, at feature density p = 1 − S = 0.3? (Hint: the positive side carries
E[xᵢ²; xᵢ > 0].)

In [ ]:
n = 3
W1 = torch.zeros(2 * n, n)
W2 = torch.zeros(n, 2 * n)
for i in range(n):
    W1[2 * i, i], W1[2 * i + 1, i] = 1.0, -1.0
    W2[i, 2 * i] = W2[i, 2 * i + 1] = 1.0
b = torch.zeros(n)

print([r.label for r in interp.neuron_roles(W1, W2)])
p = 0.3
ablation = interp.ablation_matrix(W1, W2, b, p, 20_000, make_generator(0))
print("zero-ablation Δ loss (neurons × features):")
print(ablation.zero.numpy().round(4))
print("p/6 =", round(p / 6, 4))

Zeroing a side neuron costs exactly the part of E[xᵢ²] on that side: p·½·⅓ = p/6. The other
features do not notice. That is the ablation test in its simplest form: *the role predicts
which losses move*.

Path gains do not change under the footnote-15 rescaling (W₁ row × α, W₂ column × 1/α),
because α cancels in |W₁[h,i]|·W₂[j,h]. So roles are a property of the function, not of an
arbitrary normalization.

## 2. The asymmetric superposition motif, by hand (p. 39)

Now two features share a neuron *asymmetrically*: feature 0 ("heavy") is stored with a large
input weight, feature 1 ("victim") with a small one, and the output weights are reciprocal,
so each feature's own gain is 1. On its own that would be disastrous for the victim: feature 0
interferes with output 1 four times as strongly as it should appear in output 0. A second
neuron reads a little of feature 0 and writes strongly negative to output 1, cancelling that
interference.

**Predict:** compute the four path gains of the asymmetric neuron (in → out) for inputs
W₁ = [−1, 0.25] and outputs W₂ = [1, 4]. Which feature interferes with which?

In [ ]:
W1 = torch.zeros(6, 3)
W2 = torch.zeros(3, 6)
W1[0, 0], W2[0, 0] = 1.0, 1.0  # positive side of feature 0
W1[1, 0], W1[1, 1], W2[0, 1], W2[1, 1] = -1.0, 0.25, 1.0, 4.0  # asymmetric superposition
W1[2, 1], W2[1, 2] = -1.0, 1.0  # negative side of feature 1
W1[3, 2], W1[3, 0], W2[2, 3], W2[1, 3] = 1.0, -0.2, 1.0, -25.0  # inhibitor + side of feature 2
W1[4, 2], W2[2, 4] = -1.0, 1.0  # negative side of feature 2
b = torch.zeros(3)

G = interp.path_gains(W1, W2)
print("asymmetric neuron, G[in, out]:\n", G[1, :2, :2].numpy())
print("inhibitor, G[in, out]:\n", G[3].numpy())
for role in interp.neuron_roles(W1, W2):
    print(role.neuron, role.label, "triggers:", role.triggers)
interp.find_motifs(W1, W2, b)

Compare with the p. 39 panels: "orange × red ≫ 1" is G[0→1] = 4, "red × orange ≪ 1" is
G[1→0] = 0.25. The inhibitor's G[0→1] = −5 more than cancels it: the **net interference** of
feature 0 on output 1 (summed over every neuron that fires on that side of feature 0, plus the
bias) is 4 − 5 = −1, negative, as the paper says ("converts positive interference … into
negative interference").

**Ablations tell the same story.** Zero the inhibitor and feature 1's loss should jump
(the interference is exposed), and so should feature 2's (the inhibitor is also its positive
side).

In [ ]:
ablation = interp.ablation_matrix(W1, W2, b, 0.05, 20_000, make_generator(1))
roles = interp.neuron_roles(W1, W2)
interp.role_table(roles, W1, W2, ablation)

## 3. Does gradient descent find it? (E09)

E09 trains |x| models with m = 2n − 2 neurons, two short of the 2n that computing |x| without
superposition needs, like the p. 39 model (6 features, 10 neurons). At a moderate density the
model must squeeze one feature. The run then does to every best model what we just did by
hand, and searches every restart for the motif.

**Predict:** which feature will be the victim, the least or the most important one? (Here all
features have equal importance, so: will it be the same feature across restarts?)

In [ ]:
run = quick_run("mech_interp_abs")
criteria(run)

In [ ]:
display(Image(filename=str(run.figures_dir / "motif_model.png")))
display(Image(filename=str(run.figures_dir / "ablations.png")))

In [ ]:
summary = run.read_table("metrics")
summary[
    [
        "shape",
        "feature_probability",
        "graded",
        "named",
        "live",
        "agree",
        "motifs",
        "restarts_with_motif",
        "rebuild_recovered",
    ]
]

Look at the context rows (not graded): at the denser setting no motif appears (the model just
drops a feature and keeps clean side neurons), while at the sparsest one neurons that fit no
role ("unexplained") can appear, and then the rebuild recovers less. The motif lives in a band of densities where
superposition is *limited*, the regime p. 39 describes.

In [ ]:
display(Image(filename=str(run.figures_dir / "role_census.png")))
display(Image(filename=str(run.figures_dir / "rebuild.png")))

## 4. Your turn: analyze one trained model yourself

Load one best model from the run and run the pipeline: roles, motifs, ablations, rebuild.

In [ ]:
from superposition.experiments.mech_interp_abs import TrainedShape

trained = TrainedShape(**run.load_checkpoint("n6_m10"))
condition = 0  # the first graded density
k = int(trained.best[condition])
W1, W2, b = trained.model_weights(k)
p = trained.probabilities[condition]
roles = interp.neuron_roles(W1, W2)
print(f"1 − S = {p}:", [r.label for r in roles])
for motif in interp.find_motifs(W1, W2, b):
    print(motif)
rebuild = interp.rebuild_test(roles, W1, W2, b, trained.importance, p, 20_000, make_generator(2))
print("rebuild recovers", round(rebuild.fraction_recovered, 3), "of the loss reduction")
pd.DataFrame(
    {
        "trained": rebuild.trained,
        "rebuild": rebuild.rebuild,
        "mean prediction": rebuild.mean_prediction,
    }
).round(5)

## Exercises

**1.** The paper's example stores two features with W₁ = [2, −½] and reads them out with
W₂ = [½, 2]. Compute the four path gains. By what factor does the heavy feature's interference
on the victim exceed the reverse?

<details><summary>Solution</summary>

G = |W₁|ᵀ W₂ per pair: self gains 2·½ = 1 and ½·2 = 1; heavy → victim 2·2 = 4; victim → heavy
½·½ = ¼. The ratio is 16, the square of the size flip (4 × 4).
</details>

**2.** Why must the inhibitor read the heavy feature *with the same sign* as the asymmetric
neuron does?

<details><summary>Solution</summary>

A ReLU neuron fires on one side of each input feature. The interference only happens when the
heavy feature is on the side the asymmetric neuron reads, so the inhibitor has to fire on
that same side; reading the other side would inhibit the victim when nothing needs
cancelling (`find_motifs` checks `sign(W₁[h′, i]) == sign(W₁[h, i])`).
</details>

**3.** In the hand-built model, zero the inhibitor's trigger weight `W1[3, 0]` and recompute
the ablation table. Which feature's loss changes, and why?

<details><summary>Solution</summary>

```python
W1b = W1.clone(); W1b[3, 0] = 0.0
interp.role_table(interp.neuron_roles(W1b, W2), W1b, W2,
                  interp.ablation_matrix(W1b, W2, b, 0.05, 20_000, make_generator(1)))
```

Feature 1's baseline loss rises: without the trigger nothing cancels feature 0's interference,
so output 1 reads 4|x₀| whenever the negative side of feature 0 is active. The motif detector
no longer finds a motif.
</details>

**4.** The rebuild keeps only what the roles name. Why does it recover *less* at the sparsest
context density than in the graded regime?

<details><summary>Solution</summary>

At high sparsity many neurons still compute one side of a feature but also write sizable gains
(positive or negative) to other outputs: cross-talk above the 25% "clean" bar, so they fit no
role ("unexplained"). The rebuild keeps only what the roles name and drops that cross-talk,
which by then carries part of the loss reduction (p. 39: "a few other weights this doesn't
explain … effectively small conditional biases"). Inspect `interp.path_gains` for one of those
neurons, and compare the role census with the rebuild bars.
</details>

**5.** Path gains measure 1-sparse behavior. When could they mislead you about a neuron?

<details><summary>Solution</summary>

When several features are active at once: the pre-activation is a sum, so a neuron that looks
silent for each feature alone (e.g. two negative weights) can fire, or be silenced, by a
combination. That is why the ablation check uses the real input distribution, and why the
story is least reliable at dense settings.
</details>